In [14]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
from collections import deque
import gymnasium as gym
from typing import Optional


In [15]:
DEVICE = 'cuda' if torch.cuda.is_available() \
	else 'mps' if torch.mps.is_available() \
	else 'cpu'

print("Dispositivo disponible:",  DEVICE)


Dispositivo disponible: cuda


In [16]:
class LinearQ(nn.Module):
	def __init__(self, state_dim: int):
		super().__init__()
		self.linear = nn.Linear(state_dim, 1)

	def forward(self, state):
		return self.linear(state)


In [17]:
class ActionTreeNode:
	def __init__(self, state_dim: int, depth: int = 0, max_depth: int = 5, buffer_size: int = 256):
		self.state_dim = state_dim
		self.depth = depth
		self.max_depth = max_depth

		self.is_leaf = True
		self.model = LinearQ(state_dim).to(DEVICE)
		self.buffer = deque(maxlen=buffer_size)

		self.split_feature: int
		self.split_threshold: float
		self.left: ActionTreeNode
		self.right: ActionTreeNode
	
	def route(self, state):
		if self.is_leaf:
			return self
		if state[self.split_feature] < self.split_threshold:
			return self.left.route(state)
		else:
			return self.right.route(state)


In [18]:
class LMUT:
	def __init__(self, state_dim: int, n_actions: int, max_depth: int = 5, lr: float = 1e-3, q_scale: float = 100.0):
		self.state_dim = state_dim
		self.n_actions = n_actions
		self.max_depth = max_depth
		self.lr = lr
		self.q_scale = q_scale

		self.trees: list[ActionTreeNode] = [
			ActionTreeNode(state_dim, max_depth=max_depth)
			for _ in range(n_actions)
		]

	def predict_all_actions(self, state: np.ndarray):
		state_t = torch.tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)
		
		q_vals = []
		for a in range(self.n_actions):
			leaf = self.trees[a].route(state)
			with torch.no_grad():
				q = leaf.model(state_t).item() * self.q_scale
			q_vals.append(q)
		return q_vals

	def add_transition(self, state, action: int, teacher_q: float):
		scaled_q = teacher_q / self.q_scale
		tree = self.trees[action]
		leaf = tree.route(state)

		leaf.buffer.append((
			state, 
			scaled_q
		))

	def train_leaf(self, leaf: ActionTreeNode, batch_size: int = 64):
		# print(len(leaf.buffer))
		if len(leaf.buffer) < batch_size:
			# print(":(")
			return float('inf')
		# print(":)")

		batch = random.sample(leaf.buffer, batch_size)

		optimizer = optim.SGD(leaf.model.parameters(), lr=self.lr)
		criterion = nn.MSELoss()

		total_loss = 0.0
		for state, teacher_q in batch:
			state_t = torch.tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)
			
			target = torch.tensor(teacher_q, dtype=torch.float32, device=DEVICE)

			pred = leaf.model(state_t).squeeze()
			
			loss = criterion(pred, target)

			optimizer.zero_grad()
			loss.backward()
			optimizer.step()

			total_loss += loss.item()

		avg_loss = total_loss / batch_size
		return avg_loss
	
	def _compute_variance(self, leaf: ActionTreeNode):
		if len(leaf.buffer) <= 0:
			return 0.0
		q_vals = [q for _, q in leaf.buffer]
		return np.var(q_vals)

	def _try_split(self, node: ActionTreeNode, min_improvement: float = 0.03, batch_size: int = 64):
		# if node.depth >= node.max_depth:
			# return False
		
		if len(node.buffer) < batch_size:
			return False

		states = np.array([s for s, _ in node.buffer])
		q_vals = np.array([q for _, q in node.buffer])

		parent_var = np.var(q_vals)

		best_gain = -1.0
		best_feature = None
		best_thresh = None

		n_features = self.state_dim
		for feat in range(n_features):
			feat_vals = states[:, feat]

			thresholds = np.percentile(feat_vals, [25, 50, 75])

			for thresh in thresholds:
				left_mask = feat_vals < thresh
				right_mask = feat_vals >= thresh

				if left_mask.sum() <= 0 or right_mask.sum() <= 0:
					continue
				
				left_var = np.var(q_vals[left_mask])
				right_var = np.var(q_vals[right_mask])

				w_left = left_mask.sum() / len(q_vals)
				w_right = right_mask.sum() / len(q_vals)

				split_var = w_left * left_var + w_right * right_var

				gain = parent_var - split_var

				if gain > best_gain:
					best_gain = gain
					best_feature = feat
					best_thresh = thresh

		# print(best_gain)
		if best_gain > min_improvement:
			node.is_leaf = False
			node.split_feature = best_feature
			node.split_threshold = best_thresh

			node.left = ActionTreeNode(node.state_dim, node.depth + 1, node.max_depth)
			node.right = ActionTreeNode(node.state_dim, node.depth + 1, node.max_depth)

			node.left.model.load_state_dict(node.model.state_dict())
			node.right.model.load_state_dict(node.model.state_dict())

			for state, q_val in node.buffer:
				if state[best_feature] < best_thresh:
					node.left.buffer.append((state, q_val))
				else:
					node.right.buffer.append((state, q_val))
			
			node.buffer.clear()

			self.train_leaf(node.left, batch_size=batch_size)
			self.train_leaf(node.right, batch_size=batch_size)

			return True
		return False

	def update_all_leaves(self):
		def traverse(node: ActionTreeNode):
			if node.is_leaf:
				loss = self.train_leaf(node)
				# if loss <= 0.05:
				split_occurred = self._try_split(node)
				# else:
					# split_occurred = False

				if loss != float('inf'):
					return 1, loss, 1, 1 if split_occurred else 0
				else:
					return 1, 0.0, 0, 1 if split_occurred else 0
				
			else:
				left_lc, left_loss, left_trained, left_splits = traverse(node.left)
				right_lc, right_loss, right_trained, right_splits = traverse(node.right)
				return (left_lc + right_lc, 
						left_loss + right_loss, 
						left_trained + right_trained, 
						left_splits + right_splits)

		total_leaves = 0
		total_loss = 0
		total_trained = 0
		total_splits = 0
		for tree in self.trees:
			lc, loss_sum, trained, splits = traverse(tree)
			total_leaves += lc
			total_loss += loss_sum
			total_trained += trained
			total_splits += splits

		avg_loss = total_loss / total_trained if total_trained > 0 else 0.0
		return {'leaf_count': total_leaves, 'avg_loss': avg_loss, 'splits': total_splits}
	
	def print_tree(self, action: int, node: Optional[ActionTreeNode] = None, indent: str = ""):
		if node is None:
			print(f"\n=== Tree for action {action} ===")
			node = self.trees[action]
		
		if node.is_leaf:
			weights = node.model.linear.weight.detach().cpu().numpy().flatten()
			bias = node.model.linear.bias.detach().cpu().item()
			print(f"{indent}[Leaf] depth={node.depth} | y = {weights} * s + {bias:.4f}")
		else:
			print(f"{indent}[Node] depth={node.depth} | split: feature {node.split_feature} < {node.split_threshold:.4f}")
			print(f"{indent}  left:")
			self.print_tree(action, node.left, indent + "    ")
			print(f"{indent}  right:")
			self.print_tree(action, node.right, indent + "    ")

	def compute_current_mse(self):
		total_se = 0.0
		total_n = 0
		for tree in self.trees:
			stack = [tree]
			leaves: list[ActionTreeNode] = []
			while stack:
				node = stack.pop()
				if node.is_leaf:
					leaves.append(node)
				else:
					stack.append(node.left)
					stack.append(node.right)
			for leaf in leaves:
				for state, scaled_q in leaf.buffer:
					state_t = torch.tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)
					with torch.no_grad():
						pred_scaled = leaf.model(state_t).item()
					pred_orig = pred_scaled * self.q_scale
					target_orig = scaled_q * self.q_scale
					total_se += (pred_orig - target_orig) ** 2
					total_n += 1
		return total_se / total_n if total_n > 0 else 0.0


In [19]:
env = gym.make('CartPole-v1')
state_dim = env.observation_space.shape[0]
n_actions = env.action_space.n

print(state_dim, n_actions)


4 2


In [20]:
class DQNNetwork(nn.Module):
	"""
	Red neuronal feedforward para aproximar Q-values.
	
	Arquitectura:
	- Capa de entrada: recibe el estado del entorno
	- 2 capas ocultas Linear con activación ReLU
	- Capa de salida Linear: devuelve Q-value para cada acción posible
	
	Parámetros
	----------
	state_size : int
		Dimensión del espacio de estados (número de features de observación)
	action_size : int
		Número de acciones posibles
	hidden_size : int, opcional
		Número de neuronas en las capas ocultas (default: 128)
	"""
	
	def __init__(self, state_size: int, action_size: int, hidden_size: int = 128):
		super(DQNNetwork, self).__init__()

		# Capa de tipo secuencial que contiene las capas de la red
		# Lineal + ReLU + Lineal + ReLU + Lineal
		self.seq = nn.Sequential(
			nn.Linear(state_size, hidden_size),
			nn.ReLU(),
			nn.Linear(hidden_size, hidden_size),
			nn.ReLU(),
			nn.Linear(hidden_size, action_size)
		)
		
	def forward(self, state: torch.Tensor) -> torch.Tensor:
		"""
		Forward pass de la red.
		
		Parámetros
		----------
		state : torch.Tensor
			Estado(s) del entorno. Shape: (batch_size, state_size) o (state_size,)
			
		Retorna
		-------
		torch.Tensor
			Q-values para cada acción. Shape: (batch_size, action_size) o (action_size,)
		"""
		# Aplica la red neuronal al estado de entrada y devuelve el resultado
		return self.seq(state)


In [21]:
def epsilon_greedy_policy(policy_net: DQNNetwork, state: np.ndarray, epsilon: float, action_size: int) -> int:
	"""
	Selecciona una acción usando epsilon-greedy policy.
	
	Con probabilidad epsilon elige una acción aleatoria (exploración), y con probabilidad 
	1-epsilon elige la  mejor acción según la red (explotación).
	
	Parámetros
	----------
	policy_net : DQNNetwork
		Red neuronal que representa la política del agente
	state : np.ndarray
		Estado actual del entorno como array
	epsilon : float
		Probabilidad de elegir una acción aleatoria
	action_size : int
		Número de acciones disponibles en el entorno
		
	Retorna
	-------
	int
		Acción seleccionada
	"""
	# Exploración: acción aleatoria
	if np.random.random() < epsilon:
		# Devolver acción aleatoria con np.random.randint
		return np.random.randint(action_size)
	
	# Explotación: mejor acción según Q-values
	with torch.no_grad():
		# Transformar el estado a un tensor
		state = torch.FloatTensor(state).unsqueeze(0).to(DEVICE)

		# aplicar la red al estado para obtener los valores q
		q_values = policy_net.forward(state)
		
		# devolver el índice del mayor valor Q (como int)
		return q_values.argmax().item()
	

In [22]:
def play_episode(env: gym.Env, policy_net: DQNNetwork):
	"""
	Simula un episodio usando una política e-greedy.

	Parámetros:
		env: entorno de gym.
		policy_net: red que define la política del agente.

	Devuelve:
		Tupla (recompensa_total, pasos_totales) .
	"""
	state, info = env.reset()
	episode_steps = episode_reward = 0
	done = False
	while not done:
		# Selecciona la acción adecuada usando la función epsilon_greedy_policy
		# IMPORTANTE: epsilon tiene que ser 0 para que no haga acciones aleatorias
		action = epsilon_greedy_policy(policy_net,state, 0, env.action_space.n)

		next_state, reward, terminated, truncated, info = env.step(action)
		
		episode_steps += 1
		episode_reward += reward
		done = terminated or truncated
		state = next_state
	return episode_reward, episode_steps


In [23]:
final_model_path = 'models/dqn_cartpole'

policy_net = torch.load(final_model_path, weights_only=False)
policy_net.eval()


DQNNetwork(
  (seq): Sequential(
    (0): Linear(in_features=4, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): ReLU()
    (4): Linear(in_features=128, out_features=2, bias=True)
  )
)

In [24]:
episode_reward, episode_steps = play_episode(env, policy_net)
print(f"Recompensa del episodio: {episode_reward}")
print(f"Pasos del episodio: {episode_steps}")


Recompensa del episodio: 500.0
Pasos del episodio: 500


In [25]:
def active_play_mimic(env, teacher: DQNNetwork, mimic_lmut: LMUT, episodes: int = 200, batch_size: int = 100, epsilon_start: float = 1.0, epsilon_end: float = 0.01, decay: float = 0.995):
	epsilon = epsilon_start
	all_transitions = []
	recent_rewards = []

	for ep in range(episodes):
		episode_reward = 0
		state, _ = env.reset()
		done = False
		while not done:
			action = epsilon_greedy_policy(teacher, state, epsilon, env.action_space.n)
			
			state_t = torch.tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)
			with torch.no_grad():
				q_all = teacher.forward(state_t)
			teacher_q = q_all[0, action].item()
			# print(teacher_q)

			all_transitions.append((state, action, teacher_q))

			next_state, reward, terminated, truncated, _ = env.step(action)
			episode_reward += reward
			done = terminated or truncated
			state = next_state

			if len(all_transitions) >= batch_size:
				for s, a, tq in all_transitions:
					mimic_lmut.add_transition(s, a, tq)
				stats = mimic_lmut.update_all_leaves()
				current_mse = mimic_lmut.compute_current_mse()
				all_transitions.clear()

				print(
                    f"Episode {ep+1} | "
                    f"Leaves: {stats['leaf_count']} | "
                    f"AvgLoss: {stats['avg_loss']:.4f} | "
                    f"MSE: {current_mse:.4f} | "
                    f"Splits: {stats['splits']} | "
					f"ε: {epsilon:.3f}"
                )
				
		recent_rewards.append(episode_reward)
		if len(recent_rewards) > 100:
			recent_rewards.pop(0)
		avg_reward = sum(recent_rewards) / len(recent_rewards)

		# print(
		# 	f"Episode {ep+1:4d} | Reward: {episode_reward:6.1f} | "
		# 	f"Avg100: {avg_reward:6.2f} | ε: {epsilon:.3f}"
		# )
				
		epsilon = max(epsilon_end, epsilon * decay)
		
	if all_transitions:
		for s, a, tq in all_transitions:
			mimic_lmut.add_transition(s, a, tq)
		mimic_lmut.update_all_leaves()


In [26]:
mimic = LMUT(state_dim, n_actions, max_depth=5)

active_play_mimic(env, policy_net, mimic, episodes=400, batch_size=200)


Episode 8 | Leaves: 2 | AvgLoss: 13.7833 | MSE: 92735.4834 | Splits: 2 | ε: 0.966
Episode 18 | Leaves: 4 | AvgLoss: 8.6714 | MSE: 63541.2419 | Splits: 2 | ε: 0.918
Episode 24 | Leaves: 6 | AvgLoss: 6.0466 | MSE: 48987.1509 | Splits: 2 | ε: 0.891
Episode 30 | Leaves: 8 | AvgLoss: 4.7597 | MSE: 38303.0562 | Splits: 2 | ε: 0.865
Episode 37 | Leaves: 10 | AvgLoss: 3.7941 | MSE: 30209.7611 | Splits: 4 | ε: 0.835
Episode 44 | Leaves: 14 | AvgLoss: 3.0976 | MSE: 23951.5202 | Splits: 3 | ε: 0.806
Episode 49 | Leaves: 17 | AvgLoss: 2.5233 | MSE: 20501.6376 | Splits: 1 | ε: 0.786
Episode 53 | Leaves: 18 | AvgLoss: 2.0533 | MSE: 16785.3208 | Splits: 3 | ε: 0.771
Episode 60 | Leaves: 21 | AvgLoss: 1.4283 | MSE: 14643.0311 | Splits: 1 | ε: 0.744
Episode 64 | Leaves: 22 | AvgLoss: 1.0394 | MSE: 13197.1281 | Splits: 1 | ε: 0.729
Episode 69 | Leaves: 23 | AvgLoss: 0.7730 | MSE: 12295.5424 | Splits: 0 | ε: 0.711
Episode 75 | Leaves: 23 | AvgLoss: 1.2881 | MSE: 12346.0237 | Splits: 1 | ε: 0.690
Episode 

In [27]:
mimic.print_tree(0)



=== Tree for action 0 ===
[Node] depth=0 | split: feature 2 < -0.0482
  left:
    [Node] depth=1 | split: feature 0 < -0.0078
      left:
        [Node] depth=2 | split: feature 0 < -0.1768
          left:
            [Leaf] depth=3 | y = [-0.064408    0.13182642  0.35214224 -0.02588786] * s + 1.3559
          right:
            [Node] depth=3 | split: feature 2 < -0.1226
              left:
                [Leaf] depth=4 | y = [-0.07677259  0.08251221  0.32749823 -0.05726065] * s + 1.6387
              right:
                [Node] depth=4 | split: feature 3 < -0.5417
                  left:
                    [Leaf] depth=5 | y = [-0.08786664  0.03959394  0.3063748  -0.07268967] * s + 1.9266
                  right:
                    [Leaf] depth=5 | y = [-0.08786664  0.03959394  0.3063748  -0.07268967] * s + 1.9266
      right:
        [Node] depth=2 | split: feature 3 < -0.9254
          left:
            [Node] depth=3 | split: feature 3 < -1.6551
              left:
         

In [32]:
def evaluate_fidelity(env, teacher: DQNNetwork, mimic_lmut: LMUT, num_samples: int = 1000):
    mae_sum = 0.0
    mse_sum = 0.0
    count = 0
    correct = 0

    state, _ = env.reset()
    for _ in range(num_samples):
        state_t = torch.tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        with torch.no_grad():
            q_all = teacher.forward(state_t)
            print(q_all)
            teacher_action = torch.argmax(q_all[0]).item()
            teacher_q = q_all[0, teacher_action].item()

        mimic_qs = mimic_lmut.predict_all_actions(state)
        mimic_q = mimic_qs[teacher_action]
        
        mimic_action = np.argmax(mimic_qs)

        if teacher_action == mimic_action:
            correct += 1

        error = teacher_q - mimic_q
        mae_sum += abs(error)
        mse_sum += error ** 2
        count += 1

        next_state, _, terminated, truncated, _ = env.step(teacher_action)
        if terminated or truncated:
            state, _ = env.reset()
        else:
            state = next_state

    mae = mae_sum / count
    rmse = (mse_sum / count) ** 0.5
    accuracy = correct / count
    return mae, rmse, accuracy

In [33]:
mae, rmse, acc = evaluate_fidelity(env, policy_net, mimic, num_samples=10000)
print(f"Fidelity: MAE = {mae:.4f}, RMSE = {rmse:.4f}, Action Accuracy = {acc:.2%}")


tensor([[444.8174, 435.5460]], device='cuda:0')
tensor([[447.5482, 440.6434]], device='cuda:0')
tensor([[449.0717, 446.5051]], device='cuda:0')
tensor([[448.2058, 449.8441]], device='cuda:0')
tensor([[453.9354, 453.5522]], device='cuda:0')
tensor([[424.8473, 443.7998]], device='cuda:0')
tensor([[436.8126, 444.9894]], device='cuda:0')
tensor([[447.7027, 448.3314]], device='cuda:0')
tensor([[448.3247, 450.9612]], device='cuda:0')
tensor([[452.7460, 458.5477]], device='cuda:0')
tensor([[455.1109, 460.2277]], device='cuda:0')
tensor([[454.8387, 451.6195]], device='cuda:0')
tensor([[457.9532, 458.7413]], device='cuda:0')
tensor([[447.9290, 436.4267]], device='cuda:0')
tensor([[452.7581, 449.8242]], device='cuda:0')
tensor([[450.5174, 447.8920]], device='cuda:0')
tensor([[451.2920, 448.9961]], device='cuda:0')
tensor([[453.4129, 452.1539]], device='cuda:0')
tensor([[444.5164, 447.1670]], device='cuda:0')
tensor([[452.6271, 454.6589]], device='cuda:0')
tensor([[453.3357, 455.4872]], device='c

In [34]:
def evaluate(env, agent: LMUT, episodes: int = 10):
	total_reward = 0
	for _ in range(episodes):
		state, _ = env.reset()
		done = False
		while not done:
			q_vals = agent.predict_all_actions(state)
			action = np.argmax(q_vals)
			next_state, reward, terminated, truncated, _ = env.step(action)
			done = terminated or truncated
			total_reward += reward
	return total_reward / episodes


In [35]:
reward = evaluate(env, mimic, 100)
print("Mimic average return:", reward)


Mimic average return: 9.32
